# NeqSim CO2 Impurity Kinetics & Thermodynamics Interactive Guide

Welcome to the comprehensive guide for simulating **chemical reactions and phase behavior of trace impurities in dense-phase and supercritical CO2 transport streams** (ship and pipeline transport).

### Key Topics Covered in this Notebook:
1. **Model Initialization & Importing Framework**
2. **Selecting Equation of State (EOS) & Fluid Thermodynamics**
3. **Setting Impurity Levels, Water Content & Wall Materials**
4. **Running Dynamic ODE Simulations & Building Summary Tables**
   - **Table 1**: 10-Hour Species Concentration Time-Series Table
   - **Table 2**: Reaction Kinetics & Thermodynamic Equilibrium Summary Table ($E_a, K_{\text{eq}}, k_f, r_0$)
5. **Interactive Benchmark Test Cases Suite** (Cases 1–4, Oxidant-Free Streams, Pipeline Conditions & High H2S Streams)



> **Experimental model — calibration required.** Use these tutorials for research and teaching only; do not treat their parameters or results as validated design data.


## 1. Importing NeqSim Impurity Kinetics Framework


In [1]:
import numpy as np
import pandas as pd
from neqsim_co2_kinetics import CO2ImpurityKineticsModel

print("CO2 Impurity Kinetics Engine successfully imported!")


CO2 Impurity Kinetics Engine successfully imported!


## 2. Choosing Equation of State (EOS) & Calculating Fluid Density

The framework integrates with **NeqSim SRK EOS** or EOS dense fluid density correlations.
Fluid molar density $\rho_m$ (kmol/m³) is automatically calculated based on system pressure $P$ (bar) and temperature $T$ (K).

Let's test fluid properties at **Pipeline Transport Conditions** ($25^\circ\text{C}, 100\text{ bar}$) vs **Ship Transport Conditions** ($-25^\circ\text{C}, 25\text{ bar}$):



In [2]:
# Pipeline Transport (25 °C, 100 bar)
model_pipe = CO2ImpurityKineticsModel(T_kelvin=298.15, P_bar=100.0, water_ppm=50.0)
print(f"Pipeline Molar Density (25 °C, 100 bar):   {model_pipe.molar_density:.2f} kmol/m³")

# Ship Transport (-25 °C, 25 bar)
model_ship = CO2ImpurityKineticsModel(T_kelvin=248.15, P_bar=25.0, water_ppm=50.0)
print(f"Ship Transport Molar Density (-25 °C, 25 bar): {model_ship.molar_density:.2f} kmol/m³")


Pipeline Molar Density (25 °C, 100 bar):   20.66 kmol/m³
Ship Transport Molar Density (-25 °C, 25 bar): 24.03 kmol/m³


## 3. Setting Impurity Levels & Choosing Wall Material

You can specify arbitrary trace impurity concentrations in **parts per million (ppm)**:
- `H2S`: Hydrogen Sulfide (ppm)
- `SO2`: Sulfur Dioxide (ppm)
- `NO2`: Nitrogen Dioxide (ppm)
- `O2`: Molecular Oxygen (ppm)
- `H2O`: Water Content (ppm)

### Wall Material Options:
- `'carbon_steel'` or `'magnetite'`: Catalyzes heterogeneous elemental sulfur formation ($R_8$: $E_{a, \text{S8}} = 42.0\text{ kJ/mol}$).
- `'stainless_steel'` or `'inert'`: Uncatalyzed surface ($E_{a, \text{S8}} = 65.0\text{ kJ/mol}$).



In [3]:
# Define custom feed stream in ppm
custom_feed = {"H2S": 10.0, "SO2": 10.0, "NO2": 10.0, "O2": 10.0, "H2O": 10.0}

# Select Carbon Steel / Magnetite Surface
model = CO2ImpurityKineticsModel(
    T_kelvin=248.15,  # -25 °C
    P_bar=25.0,  # 25 bar
    water_ppm=10.0,  # 10 ppm H2O
    material="carbon_steel",
)

print(f"Model initialized for material: '{model.material}' at T={model.T} K, P={model.P} bar")


Model initialized for material: 'carbon_steel' at T=248.15 K, P=25.0 bar


## 4. Helper Functions for Generating Table 1 & Table 2

Below are standard helper functions to simulate any case and format both **Table 1 (Time-Series)** and **Table 2 (Reaction Kinetics & $K_{\text{eq}}$)**.



In [4]:
def generate_tables_for_case(case_name, feed, T_K=248.15, P_bar=25.0, material="carbon_steel"):
    model = CO2ImpurityKineticsModel(
        T_kelvin=T_K, P_bar=P_bar, water_ppm=feed.get("H2O", 10.0), material=material
    )
    rho_m = model.molar_density

    # 1. Dynamic ODE Integration over 10 hours
    res = model.simulate(feed, duration_sec=10.0 * 3600.0, num_points=1001)
    t_h = res["time_hours"]

    # Build Table 1 (Concentration Time-Series)
    target_hours = [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0]
    table1_rows = []

    for target in target_hours:
        idx = np.argmin(np.abs(t_h - target))
        row = {
            "Time (h)": target,
            "H2S (ppm)": res["ppm"]["H2S"][idx],
            "SO2 (ppm)": res["ppm"]["SO2"][idx],
            "NO2 (ppm)": res["ppm"]["NO2"][idx],
            "NO (ppm)": res["ppm"]["NO"][idx],
            "O2 (ppm)": res["ppm"]["O2"][idx],
            "H2O (ppm)": res["ppm"]["H2O"][idx],
            "H2SO4 (ppm)": res["ppm"]["H2SO4"][idx],
            "HNO3 (ppm)": res["ppm"]["HNO3"][idx],
            "NH3 (ppm)": res["ppm"]["NH3"][idx],
            "S8 (ppm)": res["ppm"]["S8"][idx],
        }
        table1_rows.append(row)

    df_table1 = pd.DataFrame(table1_rows)

    # Build Table 2 (Reaction Kinetics & Keq Summary)
    rates_dict = model._calculate_pure_physical_rate_constants(feed.get("H2O", 10.0))
    reactions_info = [
        (
            "R1",
            "SO2 + 0.5 O2 + H2O <-> H2SO4",
            "Direct Thermal SO2 Oxidation",
            45.0,
            rates_dict["k1_f"],
            rates_dict["Keq1"],
        ),
        (
            "R2",
            "H2S + 3 NO2 <-> SO2 + H2O + 3 NO",
            "H2S Oxidation by NO2",
            28.0,
            rates_dict["k2_f"],
            rates_dict["Keq2"],
        ),
        (
            "R3a",
            "SO2 + NO2 + H2O <-> NO + H2SO4",
            "Base NO2 Oxidation (No H2S)",
            26.0,
            rates_dict["k3a_f"],
            rates_dict["Keq3"],
        ),
        (
            "R3b",
            "SO2 + H2S + NO2 + O2 -> H2SO4",
            "Radical Chain Co-Catalysis",
            15.0,
            rates_dict["k3b_f"],
            rates_dict["Keq3"],
        ),
        (
            "R4",
            "2 NO + O2 <-> 2 NO2",
            "NO Termolecular Re-Oxidation",
            -4.4,
            rates_dict["k4_f"],
            rates_dict["Keq4"],
        ),
        (
            "R5",
            "3 NO2 + H2O <-> 2 HNO3 + NO",
            "Reversible NO2 Hydrolysis",
            28.0,
            rates_dict["k5_f"],
            rates_dict["Keq5"],
        ),
        (
            "R6",
            "H2S + 1.5 O2 <-> SO2 + H2O",
            "Uncatalyzed Direct H2S Oxidation",
            65.0,
            rates_dict["k6_f"],
            rates_dict["Keq6"],
        ),
        (
            "R7",
            "5 H2S + 6 NO + 4 H2O -> 6 NH3 + 5 SO2",
            "Trace Ammonia Reactions",
            15.0,
            rates_dict["k7_f"],
            1e10,
        ),
        (
            "R8",
            "H2S + 0.5 O2 -> 1/8 S8 + H2O",
            "Catalytic S8 Formation (CS/Magnetite)",
            rates_dict["Ea8"],
            rates_dict["k8_f"],
            1e10,
        ),
    ]

    C_H2S = (feed.get("H2S", 1e-6) * 1e-6) * rho_m
    C_SO2 = (feed.get("SO2", 1e-6) * 1e-6) * rho_m
    C_NO2 = (feed.get("NO2", 1e-6) * 1e-6) * rho_m
    C_NO = (feed.get("NO", 1e-6) * 1e-6) * rho_m
    C_O2 = (feed.get("O2", 1e-6) * 1e-6) * rho_m
    C_H2O = (feed.get("H2O", 1e-6) * 1e-6) * rho_m

    table2_rows = []
    for rxn_id, eq, name, ea, k_f, keq in reactions_info:
        if rxn_id == "R1":
            r0 = k_f * C_SO2 * (C_O2**0.5) * C_H2O
        elif rxn_id == "R2":
            r0 = k_f * C_H2S * C_NO2
        elif rxn_id == "R3a":
            r0 = k_f * C_SO2 * C_NO2 * C_H2O
        elif rxn_id == "R3b":
            r0 = k_f * C_SO2 * (C_H2S**0.5) * C_NO2 * (C_O2**0.5)
        elif rxn_id == "R4":
            r0 = k_f * (C_NO**2) * C_O2
        elif rxn_id == "R5":
            r0 = k_f * (C_NO2**3) * C_H2O
        elif rxn_id == "R6":
            r0 = k_f * C_H2S * (C_O2**0.5)
        elif rxn_id == "R7":
            r0 = k_f * C_H2S * C_NO * C_H2O
        elif rxn_id == "R8":
            r0 = k_f * C_H2S * (C_O2**0.5)

        r0_ppm_hr = (r0 / rho_m) * 1e6 * 3600.0
        table2_rows.append(
            {
                "ID": rxn_id,
                "Reaction Equation": eq,
                "Ea (kJ/mol)": ea,
                "Keq": f"{keq:.4e}",
                "k_f": f"{k_f:.4e}",
                "r0 (kmol/m3.s)": f"{r0:.4e}",
                "r0 (ppm/hr)": f"{r0_ppm_hr:.4e}",
            }
        )

    df_table2 = pd.DataFrame(table2_rows)

    print("=" * 100)
    print(f"{case_name.upper()} (T={T_K - 273.15:.1f} °C, P={P_bar} bar, Material={material})")
    print("=" * 100)
    print("\nTABLE 1: 10-HOUR SPECIES CONCENTRATION TIME-SERIES TABLE")
    display(df_table1)

    print("\nTABLE 2: REACTION KINETICS & THERMODYNAMIC EQUILIBRIUM SUMMARY TABLE")
    display(df_table2)

    return df_table1, df_table2


## 5. Interactive Benchmark Test Cases Suite

Below you can run any of the test cases examined together by executing the corresponding cell:



### Example 1: Case 1 Baseline (With BOTH H2S & NO2 at -25 °C, 25 bar)


In [5]:
feed_case1 = {"H2S": 10.0, "SO2": 10.0, "NO2": 10.0, "O2": 10.0, "H2O": 10.0}
df1_t1, df1_t2 = generate_tables_for_case(
    "Case 1: Baseline with Both H2S & NO2", feed_case1, T_K=248.15, P_bar=25.0
)


CASE 1: BASELINE WITH BOTH H2S & NO2 (T=-25.0 °C, P=25.0 bar, Material=carbon_steel)

TABLE 1: 10-HOUR SPECIES CONCENTRATION TIME-SERIES TABLE

TABLE 2: REACTION KINETICS & THERMODYNAMIC EQUILIBRIUM SUMMARY TABLE


,Time (h),H2S (ppm),SO2 (ppm),NO2 (ppm),NO (ppm),O2 (ppm),H2O (ppm),H2SO4 (ppm),HNO3 (ppm),NH3 (ppm),S8 (ppm)
0,0.0,1.000000e+01,10.000000,10.000000,0.000000,10.000000,10.000000,0.000000,0.000000e+00,0.000000,0.000000
1,1.0,8.408984e-01,13.615186,0.001056,3.871226,8.918815,4.426030,5.541494,8.003738e-11,6.127718,0.000303
2,2.0,2.129515e-01,12.424023,0.003153,3.461352,8.483656,2.623692,7.360114,8.003738e-11,6.535495,0.000364
3,3.0,8.336238e-07,9.663202,0.177378,3.253248,8.118783,0.000000,10.333808,8.003862e-11,6.569374,0.000374
4,4.0,0.000000e+00,9.657545,0.710063,2.720563,7.852439,0.000000,10.339466,8.003862e-11,6.569374,0.000374
5,5.0,0.000000e+00,9.657545,1.083625,2.347001,7.665658,0.000000,10.339467,8.003862e-11,6.569374,0.000374
6,6.0,0.000000e+00,9.657544,1.361795,2.068831,7.526573,0.000000,10.339467,8.003862e-11,6.569374,0.000374
7,7.0,0.000000e+00,9.657544,1.577846,1.852780,7.418547,0.000000,10.339467,8.003862e-11,6.569374,0.000374
8,8.0,0.000000e+00,9.657544,1.750983,1.679643,7.331979,0.000000,10.339467,8.003862e-11,6.569374,0.000374
9,9.0,0.000000e+00,9.657544,1.893130,1.537496,7.260905,0.000000,10.339467,8.003862e-11,6.569374,0.000374


,ID,Reaction Equation,Ea (kJ/mol),Keq,k_f,r0 (kmol/m3.s),r0 (ppm/hr)
0,R1,SO2 + 0.5 O2 + H2O <-> H2SO4,45.0,1.5284e+32,9.3484e-02,8.3714e-11,1.2539e-02
1,R2,H2S + 3 NO2 <-> SO2 + H2O + 3 NO,28.0,5.6861e+83,4.8444e+03,2.7982e-04,4.1914e+04
2,R3a,SO2 + NO2 + H2O <-> NO + H2SO4,26.0,5.6738e+24,1.6569e-03,2.3003e-14,3.4456e-06
3,R3b,SO2 + H2S + NO2 + O2 -> H2SO4,15.0,5.6738e+24,3.2522e+05,4.5149e-06,6.7627e+02
4,R4,2 NO + O2 <-> 2 NO2,-4.4,7.2568e+14,4.2319e+03,5.8750e-22,8.8001e-14
5,R5,3 NO2 + H2O <-> 2 HNO3 + NO,28.0,5.0811e-05,3.0650e+00,1.0227e-14,1.5318e-06
6,R6,H2S + 1.5 O2 <-> SO2 + H2O,65.0,1.1116e+106,1.6859e-04,6.2815e-10,9.4089e-02
7,R7,5 H2S + 6 NO + 4 H2O -> 6 NH3 + 5 SO2,15.0,1.0000e+10,5.9583e+03,8.2717e-15,1.2390e-06
8,R8,H2S + 0.5 O2 -> 1/8 S8 + H2O,42.0,1.0000e+10,2.1648e-05,8.0660e-11,1.2082e-02


### Example 2: Case 2 (Without H2S at -25 °C, 25 bar)


In [6]:
feed_case2 = {"H2S": 1e-6, "SO2": 10.0, "NO2": 10.0, "O2": 10.0, "H2O": 10.0}
df2_t1, df2_t2 = generate_tables_for_case("Case 2: Without H2S", feed_case2, T_K=248.15, P_bar=25.0)


CASE 2: WITHOUT H2S (T=-25.0 °C, P=25.0 bar, Material=carbon_steel)

TABLE 1: 10-HOUR SPECIES CONCENTRATION TIME-SERIES TABLE

TABLE 2: REACTION KINETICS & THERMODYNAMIC EQUILIBRIUM SUMMARY TABLE


,Time (h),H2S (ppm),SO2 (ppm),NO2 (ppm),NO (ppm),O2 (ppm),H2O (ppm),H2SO4 (ppm),HNO3 (ppm),NH3 (ppm),S8 (ppm)
0,0.0,0.000001,10.000000,10.000000,0.000000,10.000000,10.000000,0.000000,0.000000,0.000000e+00,0.000000e+00
1,1.0,0.000000,9.982581,9.999990,0.000007,9.994499,9.982580,0.017420,0.000002,2.528534e-16,3.696756e-14
2,2.0,0.000000,9.971602,9.999984,0.000011,9.989011,9.971600,0.028399,0.000005,2.528534e-16,3.696756e-14
3,3.0,0.000000,9.960650,9.999977,0.000016,9.983537,9.960647,0.039351,0.000007,2.528534e-16,3.696756e-14
4,4.0,0.000000,9.949725,9.999970,0.000020,9.978076,9.949720,0.050276,0.000010,2.528534e-16,3.696756e-14
5,5.0,0.000000,9.938827,9.999964,0.000024,9.972629,9.938821,0.061174,0.000012,2.528534e-16,3.696756e-14
6,6.0,0.000000,9.927956,9.999957,0.000028,9.967196,9.927949,0.072045,0.000015,2.528534e-16,3.696756e-14
7,7.0,0.000000,9.917112,9.999950,0.000032,9.961775,9.917103,0.082889,0.000017,2.528534e-16,3.696756e-14
8,8.0,0.000000,9.906294,9.999944,0.000036,9.956368,9.906284,0.093707,0.000020,2.528534e-16,3.696756e-14
9,9.0,0.000000,9.895503,9.999937,0.000040,9.950974,9.895492,0.104498,0.000022,2.528534e-16,3.696756e-14


,ID,Reaction Equation,Ea (kJ/mol),Keq,k_f,r0 (kmol/m3.s),r0 (ppm/hr)
0,R1,SO2 + 0.5 O2 + H2O <-> H2SO4,45.0,1.5284e+32,9.3484e-02,8.3714e-11,1.2539e-02
1,R2,H2S + 3 NO2 <-> SO2 + H2O + 3 NO,28.0,5.6861e+83,4.8444e+03,2.7982e-11,4.1914e-03
2,R3a,SO2 + NO2 + H2O <-> NO + H2SO4,26.0,5.6738e+24,1.6569e-03,2.3003e-14,3.4456e-06
3,R3b,SO2 + H2S + NO2 + O2 -> H2SO4,15.0,5.6738e+24,3.2522e+05,1.4277e-09,2.1386e-01
4,R4,2 NO + O2 <-> 2 NO2,-4.4,7.2568e+14,4.2319e+03,5.8750e-22,8.8001e-14
5,R5,3 NO2 + H2O <-> 2 HNO3 + NO,28.0,5.0811e-05,3.0650e+00,1.0227e-14,1.5318e-06
6,R6,H2S + 1.5 O2 <-> SO2 + H2O,65.0,1.1116e+106,1.6859e-04,6.2815e-17,9.4089e-09
7,R7,5 H2S + 6 NO + 4 H2O -> 6 NH3 + 5 SO2,15.0,1.0000e+10,5.9583e+03,8.2717e-22,1.2390e-13
8,R8,H2S + 0.5 O2 -> 1/8 S8 + H2O,42.0,1.0000e+10,2.1648e-05,8.0660e-18,1.2082e-09


### Example 3: Case 3 (Without NO2 at -25 °C, 25 bar)


In [7]:
feed_case3 = {"H2S": 10.0, "SO2": 10.0, "NO2": 1e-6, "O2": 10.0, "H2O": 10.0}
df3_t1, df3_t2 = generate_tables_for_case("Case 3: Without NO2", feed_case3, T_K=248.15, P_bar=25.0)


CASE 3: WITHOUT NO2 (T=-25.0 °C, P=25.0 bar, Material=carbon_steel)

TABLE 1: 10-HOUR SPECIES CONCENTRATION TIME-SERIES TABLE

TABLE 2: REACTION KINETICS & THERMODYNAMIC EQUILIBRIUM SUMMARY TABLE


,Time (h),H2S (ppm),SO2 (ppm),NO2 (ppm),NO (ppm),O2 (ppm),H2O (ppm),H2SO4 (ppm),HNO3 (ppm),NH3 (ppm),S8 (ppm)
0,0.0,10.000000,10.000000,0.000001,0.000000e+00,10.000000,10.000000,0.000000,0.000000e+00,0.000000e+00,0.000000
1,1.0,9.902516,10.075310,0.000000,1.717462e-09,9.859329,10.086402,0.011081,7.326163e-32,9.983310e-07,0.001387
2,2.0,9.806661,10.149078,0.000000,0.000000e+00,9.720865,10.171078,0.022260,7.326163e-32,1.000072e-06,0.002750
3,3.0,9.712399,10.221341,0.000000,0.000000e+00,9.584562,10.254067,0.033532,7.326163e-32,1.000142e-06,0.004091
4,4.0,9.619696,10.292132,0.000000,0.000000e+00,9.450376,10.335408,0.044895,7.326163e-32,1.000142e-06,0.005410
5,5.0,9.528520,10.361484,0.000000,0.000000e+00,9.318263,10.415135,0.056343,7.326163e-32,1.000142e-06,0.006707
6,6.0,9.438837,10.429430,0.000000,0.000000e+00,9.188179,10.493287,0.067875,7.326163e-32,1.000142e-06,0.007982
7,7.0,9.350618,10.496000,0.000000,0.000000e+00,9.060085,10.569896,0.079484,7.326163e-32,1.000142e-06,0.009237
8,8.0,9.263832,10.561226,0.000000,0.000000e+00,8.933938,10.644998,0.091169,7.326163e-32,1.000142e-06,0.010472
9,9.0,9.178448,10.625137,0.000000,0.000000e+00,8.809701,10.718625,0.102925,7.326163e-32,1.000142e-06,0.011686


,ID,Reaction Equation,Ea (kJ/mol),Keq,k_f,r0 (kmol/m3.s),r0 (ppm/hr)
0,R1,SO2 + 0.5 O2 + H2O <-> H2SO4,45.0,1.5284e+32,9.3484e-02,8.3714e-11,1.2539e-02
1,R2,H2S + 3 NO2 <-> SO2 + H2O + 3 NO,28.0,5.6861e+83,4.8444e+03,2.7982e-11,4.1914e-03
2,R3a,SO2 + NO2 + H2O <-> NO + H2SO4,26.0,5.6738e+24,1.6569e-03,2.3003e-21,3.4456e-13
3,R3b,SO2 + H2S + NO2 + O2 -> H2SO4,15.0,5.6738e+24,3.2522e+05,4.5149e-13,6.7627e-05
4,R4,2 NO + O2 <-> 2 NO2,-4.4,7.2568e+14,4.2319e+03,5.8750e-22,8.8001e-14
5,R5,3 NO2 + H2O <-> 2 HNO3 + NO,28.0,5.0811e-05,3.0650e+00,1.0227e-35,1.5318e-27
6,R6,H2S + 1.5 O2 <-> SO2 + H2O,65.0,1.1116e+106,1.6859e-04,6.2815e-10,9.4089e-02
7,R7,5 H2S + 6 NO + 4 H2O -> 6 NH3 + 5 SO2,15.0,1.0000e+10,5.9583e+03,8.2717e-15,1.2390e-06
8,R8,H2S + 0.5 O2 -> 1/8 S8 + H2O,42.0,1.0000e+10,2.1648e-05,8.0660e-11,1.2082e-02


### Example 4: Case 4 (High Water 675 ppm & High NO2 72 ppm at 25 °C, 100 bar)


In [8]:
feed_case4 = {"H2O": 675.0, "NO2": 72.0, "SO2": 10.0, "O2": 10.0, "H2S": 1e-6}
df4_t1, df4_t2 = generate_tables_for_case(
    "Case 4: High Water & NO2 Equilibrium Bound", feed_case4, T_K=298.15, P_bar=100.0
)


CASE 4: HIGH WATER & NO2 EQUILIBRIUM BOUND (T=25.0 °C, P=100.0 bar, Material=carbon_steel)

TABLE 1: 10-HOUR SPECIES CONCENTRATION TIME-SERIES TABLE

TABLE 2: REACTION KINETICS & THERMODYNAMIC EQUILIBRIUM SUMMARY TABLE


,Time (h),H2S (ppm),SO2 (ppm),NO2 (ppm),NO (ppm),O2 (ppm),H2O (ppm),H2SO4 (ppm),HNO3 (ppm),NH3 (ppm),S8 (ppm)
0,0.0,0.000001,10.000000,72.000000,0.000000,10.000000,675.000000,0.000000,0.000000,0.000000e+00,0.000000e+00
1,1.0,0.000000,2.196419,71.408594,0.212439,6.110812,667.006935,7.803582,0.378967,7.552008e-16,1.468546e-14
2,2.0,0.000000,0.594080,70.894378,0.386229,5.311432,665.234383,9.405921,0.719393,7.553227e-16,1.468546e-14
3,3.0,0.000000,0.169545,70.504402,0.514455,5.097840,664.678973,9.830456,0.981143,7.555055e-16,1.468546e-14
4,4.0,0.000000,0.049143,70.258795,0.592528,5.034792,664.474804,9.950858,1.148677,7.557295e-16,1.468546e-14
5,5.0,0.000000,0.014313,70.126221,0.631870,5.013741,664.393359,9.985688,1.241908,7.559765e-16,1.468546e-14
6,6.0,0.000000,0.004177,70.061280,0.648187,5.004675,664.358910,9.995824,1.290533,7.562342e-16,1.468546e-14
7,7.0,0.000000,0.001220,70.031020,0.652770,4.999069,664.343115,9.998781,1.316210,7.564957e-16,1.468546e-14
8,8.0,0.000000,0.000356,70.017187,0.651853,4.994491,664.334876,9.999645,1.330961,7.567579e-16,1.468546e-14
9,9.0,0.000000,0.000104,70.010831,0.648485,4.990250,664.329762,9.999897,1.340684,7.570192e-16,1.468546e-14


,ID,Reaction Equation,Ea (kJ/mol),Keq,k_f,r0 (kmol/m3.s),r0 (ppm/hr)
0,R1,SO2 + 0.5 O2 + H2O <-> H2SO4,45.0,6.1224e+26,2.7746e+00,1.1496e-07,2.0028e+01
1,R2,H2S + 3 NO2 <-> SO2 + H2O + 3 NO,28.0,5.1176e+69,5.5492e+04,1.7061e-09,2.9722e-01
2,R3a,SO2 + NO2 + H2O <-> NO + H2SO4,26.0,4.0052e+20,7.3835e-02,3.1663e-10,5.5162e-02
3,R3b,SO2 + H2S + NO2 + O2 -> H2SO4,15.0,4.0052e+20,1.4046e+06,2.8220e-08,4.9163e+00
4,R4,2 NO + O2 <-> 2 NO2,-4.4,2.3366e+12,2.9579e+03,2.6100e-22,4.5470e-14
5,R5,3 NO2 + H2O <-> 2 HNO3 + NO,28.0,2.6673e-04,2.9842e+01,1.3709e-09,2.3883e-01
6,R6,H2S + 1.5 O2 <-> SO2 + H2O,65.0,1.8279e+88,6.5360e-03,1.9415e-15,3.3824e-07
7,R7,5 H2S + 6 NO + 4 H2O -> 6 NH3 + 5 SO2,15.0,1.0000e+10,1.5802e+04,9.4119e-20,1.6397e-11
8,R8,H2S + 0.5 O2 -> 1/8 S8 + H2O,42.0,1.0000e+10,6.5767e-04,1.9536e-16,3.4035e-08


### Example 5: Oxidant-Free Stream (100 ppm H2O & 100 ppm H2S at 25 °C, 100 bar)


In [9]:
feed_oxidant_free = {"H2O": 100.0, "H2S": 100.0, "SO2": 1e-6, "NO2": 1e-6, "O2": 1e-6}
df_off_t1, df_off_t2 = generate_tables_for_case(
    "Oxidant-Free Stream", feed_oxidant_free, T_K=298.15, P_bar=100.0
)


OXIDANT-FREE STREAM (T=25.0 °C, P=100.0 bar, Material=carbon_steel)

TABLE 1: 10-HOUR SPECIES CONCENTRATION TIME-SERIES TABLE

TABLE 2: REACTION KINETICS & THERMODYNAMIC EQUILIBRIUM SUMMARY TABLE


,Time (h),H2S (ppm),SO2 (ppm),NO2 (ppm),NO (ppm),O2 (ppm),H2O (ppm),H2SO4 (ppm),HNO3 (ppm),NH3 (ppm),S8 (ppm)
0,0.0,100.000000,0.000001,0.000001,0.0,0.000001,100.0,0.000000e+00,0.000000e+00,0.000000,0.000000e+00
1,1.0,99.999998,0.000003,0.000000,0.0,0.000000,100.0,1.262179e-14,4.058269e-32,0.000001,8.138876e-09
2,2.0,99.999998,0.000003,0.000000,0.0,0.000000,100.0,1.263631e-14,4.058269e-32,0.000001,8.147542e-09
3,3.0,99.999998,0.000003,0.000000,0.0,0.000000,100.0,1.265083e-14,4.058269e-32,0.000001,8.156208e-09
4,4.0,99.999998,0.000003,0.000000,0.0,0.000000,100.0,1.266535e-14,4.058269e-32,0.000001,8.164874e-09
5,5.0,99.999998,0.000003,0.000000,0.0,0.000000,100.0,1.267988e-14,4.058269e-32,0.000001,8.173540e-09
6,6.0,99.999998,0.000003,0.000000,0.0,0.000000,100.0,1.269442e-14,4.058269e-32,0.000001,8.182205e-09
7,7.0,99.999998,0.000003,0.000000,0.0,0.000000,100.0,1.270895e-14,4.058269e-32,0.000001,8.190871e-09
8,8.0,99.999998,0.000003,0.000000,0.0,0.000000,100.0,1.272349e-14,4.058269e-32,0.000001,8.199537e-09
9,9.0,99.999998,0.000003,0.000000,0.0,0.000000,100.0,1.273803e-14,4.058269e-32,0.000001,8.208203e-09


,ID,Reaction Equation,Ea (kJ/mol),Keq,k_f,r0 (kmol/m3.s),r0 (ppm/hr)
0,R1,SO2 + 0.5 O2 + H2O <-> H2SO4,45.0,6.1224e+26,2.4930e+00,4.8390e-19,8.4303e-11
1,R2,H2S + 3 NO2 <-> SO2 + H2O + 3 NO,28.0,5.1176e+69,5.5492e+04,2.3695e-09,4.1281e-01
2,R3a,SO2 + NO2 + H2O <-> NO + H2SO4,26.0,4.0052e+20,6.6341e-02,5.8538e-26,1.0198e-17
3,R3b,SO2 + H2S + NO2 + O2 -> H2SO4,15.0,4.0052e+20,1.4046e+06,1.2394e-22,2.1593e-14
4,R4,2 NO + O2 <-> 2 NO2,-4.4,2.3366e+12,2.9579e+03,2.6100e-29,4.5470e-21
5,R5,3 NO2 + H2O <-> 2 HNO3 + NO,28.0,2.6673e-04,2.9842e+01,5.4412e-34,9.4794e-26
6,R6,H2S + 1.5 O2 <-> SO2 + H2O,65.0,1.8279e+88,6.5360e-03,6.1396e-11,1.0696e-02
7,R7,5 H2S + 6 NO + 4 H2O -> 6 NH3 + 5 SO2,15.0,1.0000e+10,1.5802e+04,1.3944e-12,2.4292e-04
8,R8,H2S + 0.5 O2 -> 1/8 S8 + H2O,42.0,1.0000e+10,6.5767e-04,6.1778e-12,1.0763e-03


### Example 6: Pipeline Transport Stream (11 ppm H2O, 69 ppm SO2, 33 ppm NO2, 180 ppm O2 at 25 °C, 70 bar)


In [10]:
feed_pipe = {"H2O": 11.0, "SO2": 69.0, "NO2": 33.0, "O2": 180.0, "H2S": 1e-6}
df_pipe_t1, df_pipe_t2 = generate_tables_for_case(
    "High-O2 Pipeline Transport Stream", feed_pipe, T_K=298.15, P_bar=70.0
)


HIGH-O2 PIPELINE TRANSPORT STREAM (T=25.0 °C, P=70.0 bar, Material=carbon_steel)

TABLE 1: 10-HOUR SPECIES CONCENTRATION TIME-SERIES TABLE

TABLE 2: REACTION KINETICS & THERMODYNAMIC EQUILIBRIUM SUMMARY TABLE


,Time (h),H2S (ppm),SO2 (ppm),NO2 (ppm),NO (ppm),O2 (ppm),H2O (ppm),H2SO4 (ppm),HNO3 (ppm),NH3 (ppm),S8 (ppm)
0,0.0,0.000001,69.000000,33.000000,0.000000,180.000000,11.000000,0.000000,0.000000,0.000000e+00,0.000000e+00
1,1.0,0.000000,66.455541,32.998672,0.000914,178.735563,8.455334,2.544460,0.000414,1.547333e-17,1.419150e-13
2,2.0,0.000000,64.568891,32.997667,0.001599,177.792530,6.568524,4.431110,0.000734,1.547333e-17,1.419150e-13
3,3.0,0.000000,63.138657,32.996895,0.002122,177.077640,5.138166,5.861344,0.000983,1.547334e-17,1.419150e-13
4,4.0,0.000000,62.041044,32.996297,0.002524,176.529013,4.040454,6.958957,0.001179,1.547335e-17,1.419150e-13
5,5.0,0.000000,61.190783,32.995831,0.002836,176.104027,3.190117,7.809218,0.001333,1.547335e-17,1.419150e-13
6,6.0,0.000000,60.527366,32.995467,0.003078,175.772436,2.526639,8.472635,0.001455,1.547336e-17,1.419150e-13
7,7.0,0.000000,60.006824,32.995181,0.003267,175.512262,2.006048,8.993177,0.001552,1.547336e-17,1.419150e-13
8,8.0,0.000000,59.596592,32.994957,0.003414,175.307227,1.595778,9.403409,0.001629,1.547336e-17,1.419150e-13
9,9.0,0.000000,59.272177,32.994781,0.003529,175.145088,1.271333,9.727824,0.001690,1.547337e-17,1.419150e-13


,ID,Reaction Equation,Ea (kJ/mol),Keq,k_f,r0 (kmol/m3.s),r0 (ppm/hr)
0,R1,SO2 + 0.5 O2 + H2O <-> H2SO4,45.0,6.1224e+26,1.1046e+00,1.7607e-08,3.3430e+00
1,R2,H2S + 3 NO2 <-> SO2 + H2O + 3 NO,28.0,5.1176e+69,5.5492e+04,6.5829e-10,1.2499e-01
2,R3a,SO2 + NO2 + H2O <-> NO + H2SO4,26.0,4.0052e+20,2.9395e-02,5.0181e-12,9.5281e-04
3,R3b,SO2 + H2S + NO2 + O2 -> H2SO4,15.0,4.0052e+20,1.4046e+06,2.9247e-07,5.5532e+01
4,R4,2 NO + O2 <-> 2 NO2,-4.4,2.3366e+12,2.9579e+03,3.6289e-21,6.8903e-13
5,R5,3 NO2 + H2O <-> 2 HNO3 + NO,28.0,2.6673e-04,2.9842e+01,1.5244e-12,2.8945e-04
6,R6,H2S + 1.5 O2 <-> SO2 + H2O,65.0,1.8279e+88,6.5360e-03,7.2394e-15,1.3746e-06
7,R7,5 H2S + 6 NO + 4 H2O -> 6 NH3 + 5 SO2,15.0,1.0000e+10,1.5802e+04,1.1848e-21,2.2495e-13
8,R8,H2S + 0.5 O2 -> 1/8 S8 + H2O,42.0,1.0000e+10,6.5767e-04,7.2846e-16,1.3831e-07


### Example 7: High H2S Pipeline Stream (60 ppm H2S, 10 ppm others at 25 °C, 100 bar)


In [11]:
feed_high_h2s = {"H2S": 60.0, "SO2": 10.0, "NO2": 10.0, "O2": 10.0, "H2O": 10.0}
df_h2s_t1, df_h2s_t2 = generate_tables_for_case(
    "High-H2S Pipeline Transport Stream", feed_high_h2s, T_K=298.15, P_bar=100.0
)


HIGH-H2S PIPELINE TRANSPORT STREAM (T=25.0 °C, P=100.0 bar, Material=carbon_steel)

TABLE 1: 10-HOUR SPECIES CONCENTRATION TIME-SERIES TABLE

TABLE 2: REACTION KINETICS & THERMODYNAMIC EQUILIBRIUM SUMMARY TABLE


,Time (h),H2S (ppm),SO2 (ppm),NO2 (ppm),NO (ppm),O2 (ppm),H2O (ppm),H2SO4 (ppm),HNO3 (ppm),NH3 (ppm),S8 (ppm)
0,0.0,60.000000,10.000000,10.0,0.0,10.0,10.000000,0.000000,0.000000e+00,0.0,0.00000
1,1.0,41.273971,27.409762,0.0,0.0,0.0,13.054325,0.671703,8.338232e-12,10.0,0.08057
2,2.0,41.273971,27.409762,0.0,0.0,0.0,13.054325,0.671703,8.338232e-12,10.0,0.08057
3,3.0,41.273971,27.409762,0.0,0.0,0.0,13.054325,0.671703,8.338232e-12,10.0,0.08057
4,4.0,41.273971,27.409762,0.0,0.0,0.0,13.054325,0.671703,8.338232e-12,10.0,0.08057
5,5.0,41.273971,27.409762,0.0,0.0,0.0,13.054325,0.671703,8.338232e-12,10.0,0.08057
6,6.0,41.273971,27.409762,0.0,0.0,0.0,13.054325,0.671703,8.338232e-12,10.0,0.08057
7,7.0,41.273971,27.409762,0.0,0.0,0.0,13.054325,0.671703,8.338232e-12,10.0,0.08057
8,8.0,41.273971,27.409762,0.0,0.0,0.0,13.054325,0.671703,8.338232e-12,10.0,0.08057
9,9.0,41.273971,27.409762,0.0,0.0,0.0,13.054325,0.671703,8.338232e-12,10.0,0.08057


,ID,Reaction Equation,Ea (kJ/mol),Keq,k_f,r0 (kmol/m3.s),r0 (ppm/hr)
0,R1,SO2 + 0.5 O2 + H2O <-> H2SO4,45.0,6.1224e+26,1.0709e+00,6.5732e-10,1.1451e-01
1,R2,H2S + 3 NO2 <-> SO2 + H2O + 3 NO,28.0,5.1176e+69,5.5492e+04,1.4217e-02,2.4768e+06
2,R3a,SO2 + NO2 + H2O <-> NO + H2SO4,26.0,4.0052e+20,2.8497e-02,2.5145e-13,4.3806e-05
3,R3b,SO2 + H2S + NO2 + O2 -> H2SO4,15.0,4.0052e+20,1.4046e+06,3.0360e-05,5.2891e+03
4,R4,2 NO + O2 <-> 2 NO2,-4.4,2.3366e+12,2.9579e+03,2.6100e-22,4.5470e-14
5,R5,3 NO2 + H2O <-> 2 HNO3 + NO,28.0,2.6673e-04,2.9842e+01,5.4412e-14,9.4794e-06
6,R6,H2S + 1.5 O2 <-> SO2 + H2O,65.0,1.8279e+88,6.5360e-03,1.1649e-07,2.0294e+01
7,R7,5 H2S + 6 NO + 4 H2O -> 6 NH3 + 5 SO2,15.0,1.0000e+10,1.5802e+04,8.3662e-14,1.4575e-05
8,R8,H2S + 0.5 O2 -> 1/8 S8 + H2O,42.0,1.0000e+10,6.5767e-04,1.1722e-08,2.0421e+00
